# 04 · Distributed training and the capstone

*Intermediate Model Training*, notebook 4 of 4 · lessons 7 and 8: **Understand distributed training and reproducibility** and **Capstone: improve a rare-class decision system**

Training on several devices at once changes where the arithmetic happens, but it must not change what is learned. In the first half you simulate data-parallel training inside one notebook: you split every batch across "workers", combine their gradients with an all-reduce, and check that the result matches training on a single worker. Then you make runs reproducible and learn what "the same result" can and cannot mean. The second half is the course capstone: a complete decision system for the rare class from notebook 1, selected on validation data, saved, reloaded and tested once.

## What you will build

- `all_reduce_mean(worker_grads, counts)`: the global-batch gradient from the workers' gradients, weighted by how many examples each worker saw;
- `data_parallel_gradients(replicas, loss_fn, X, y)`: one simulated data-parallel step, from sharding the batch to the all-reduce;
- `fingerprint(arrays)`: a short hash of a model's exact bytes, for reproducibility checks;
- `predict_with_rule(rule, X_raw)`: the saved decision rule (preprocessing, weights and threshold) applied to raw inputs.

## Learning objectives

By the end of this notebook you will be able to:

1. explain data parallelism: replicas, sharded batches, a gradient all-reduce, and why every worker still holds the whole model;
2. implement an example-weighted all-reduce and show that simulated workers reproduce single-worker training;
3. compute effective global batch sizes, and aggregate metrics over unequal or padded shards correctly;
4. check reproducibility at the level of bytes and at the level of tolerances, and say which one a comparison needs;
5. select a model and its threshold on validation data, save the complete decision rule, reload it, and evaluate it once on test data.

### How to work through this notebook

Run the cells from top to bottom. When you reach a **✍️ Your turn** cell, type the function yourself and then run the check cell beneath it:

- ✅ means your function matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not match yet. In both cases the notebook quietly uses a hidden reference version, so you can always keep going and come back later.

Each reference solution sits in a collapsed cell just above the function you type. Open it only when you are stuck. The capstone section also has **✏️ capstone choices** cells: those are yours to change for your own capstone.

## Dependencies

- **Tier 1:** NumPy for every calculation and Matplotlib for the plots, plus `hashlib` and `io` from the Python standard library. Nothing else is imported, so the notebook runs in JupyterLite in your browser, on the course JupyterHub server and in Google Colab.
- There are no real workers: the "workers" are a loop in one process, so the notebook demonstrates the arithmetic of data parallelism, not its speed. Real systems run the workers on separate devices and exchange gradients over a network.
- The networks use the tiny tensor engine and the `MLP`, `Adam` and `fit_standardiser` from Base Model Training, plus the rare-class tools from notebook 1. All are pasted into hidden recap cells.
- Nothing is downloaded and nothing is written to disk: the capstone saves its decision rule into an in-memory file.

## Setup

Run this cell first. It imports NumPy and Matplotlib, creates a seeded random generator and defines `practice`, the helper that checks each function you write against a reference.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import time

import matplotlib.pyplot as plt
import hashlib
import io

## 📦 Recap: the tiny tensor engine you built in Base Model Training

The hidden cell below is the reverse-mode autograd engine from Base Model Training, pasted unchanged: `Tensor` (with `+`, `-`, `*`, `/`, `**`, `@`, `exp`, `log`, `tanh`, `relu`, `sigmoid`, `sum`, `mean`, `reshape`, `transpose`, indexing, `log_softmax`, `softmax` and `backward`), `parameter`, `zeros_parameter`, `zero_grad`, `cross_entropy`, `layer_norm`, `numerical_grad` and `relative_error`. Every operation in it was checked against numerical gradients. Run the cell; open it only if you want to look back at the code.

In [ ]:
# ── Tiny tensor autograd (NumPy) ── the engine you built in Base Model Training ──
import numpy as np


def _unbroadcast(grad, shape):
    """Sum a gradient back down to `shape`, undoing NumPy broadcasting."""
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for axis, size in enumerate(shape):
        if size == 1 and grad.shape[axis] != 1:
            grad = grad.sum(axis=axis, keepdims=True)
    return grad


class Tensor:
    """A NumPy array that remembers how it was made, so gradients can flow back."""

    def __init__(self, data, requires_grad=False, parents=(), backward=None, op=""):
        self.data = np.asarray(data, dtype=np.float64)
        self.requires_grad = requires_grad
        self.grad = None
        self._parents = parents
        self._backward = backward  # grad_out -> one gradient per parent
        self.op = op

    shape = property(lambda self: self.data.shape)
    ndim = property(lambda self: self.data.ndim)
    T = property(lambda self: self.transpose())

    def __repr__(self):
        return f"Tensor(shape={self.shape}, op={self.op or 'leaf'}, requires_grad={self.requires_grad})"

    @staticmethod
    def lift(value):
        return value if isinstance(value, Tensor) else Tensor(value)

    def _result(self, data, parents, backward, op):
        if any(parent.requires_grad for parent in parents):
            return Tensor(data, True, parents, backward, op)
        return Tensor(data, op=op)

    # arithmetic, with broadcasting
    def __add__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data + other.data, (self, other), lambda g: (g, g), "add")

    __radd__ = __add__

    def __neg__(self):
        return self._result(-self.data, (self,), lambda g: (-g,), "neg")

    def __sub__(self, other):
        return self + (-Tensor.lift(other))

    def __rsub__(self, other):
        return Tensor.lift(other) + (-self)

    def __mul__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data * other.data, (self, other),
                            lambda g: (g * other.data, g * self.data), "mul")

    __rmul__ = __mul__

    def __truediv__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data / other.data, (self, other),
                            lambda g: (g / other.data, -g * self.data / other.data ** 2), "div")

    def __rtruediv__(self, other):
        return Tensor.lift(other) / self

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float)), "only constant exponents"
        return self._result(self.data ** exponent, (self,),
                            lambda g: (g * exponent * self.data ** (exponent - 1),), "pow")

    def __matmul__(self, other):
        other = Tensor.lift(other)

        def backward(g):
            return (g @ np.swapaxes(other.data, -1, -2), np.swapaxes(self.data, -1, -2) @ g)

        return self._result(self.data @ other.data, (self, other), backward, "matmul")

    # elementwise functions
    def exp(self):
        out = np.exp(self.data)
        return self._result(out, (self,), lambda g: (g * out,), "exp")

    def log(self):
        return self._result(np.log(self.data), (self,), lambda g: (g / self.data,), "log")

    def tanh(self):
        out = np.tanh(self.data)
        return self._result(out, (self,), lambda g: (g * (1 - out ** 2),), "tanh")

    def relu(self):
        return self._result(np.maximum(self.data, 0), (self,), lambda g: (g * (self.data > 0),), "relu")

    def sigmoid(self):
        out = 0.5 * (1 + np.tanh(0.5 * self.data))  # stable for large |x|
        return self._result(out, (self,), lambda g: (g * out * (1 - out),), "sigmoid")

    # reductions and reshaping
    def sum(self, axis=None, keepdims=False):
        def backward(g):
            if axis is not None and not keepdims:
                g = np.expand_dims(g, axis)
            return (np.broadcast_to(g, self.shape).copy(),)

        return self._result(self.data.sum(axis=axis, keepdims=keepdims), (self,), backward, "sum")

    def mean(self, axis=None, keepdims=False):
        count = self.data.size if axis is None else np.prod([self.shape[a] for a in np.atleast_1d(axis)])
        return self.sum(axis=axis, keepdims=keepdims) / count

    def reshape(self, *shape):
        return self._result(self.data.reshape(*shape), (self,), lambda g: (g.reshape(self.shape),), "reshape")

    def transpose(self, *axes):
        axes = axes or tuple(reversed(range(self.ndim)))
        inverse = np.argsort(axes)
        return self._result(self.data.transpose(axes), (self,), lambda g: (g.transpose(inverse),), "transpose")

    def __getitem__(self, index):
        def backward(g):
            full = np.zeros_like(self.data)
            np.add.at(full, index, g)  # repeated indices (an embedding row used twice) add up
            return (full,)

        return self._result(self.data[index], (self,), backward, "index")

    def log_softmax(self, axis=-1):
        shifted = self.data - self.data.max(axis=axis, keepdims=True)
        out = shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))
        probs = np.exp(out)
        return self._result(out, (self,), lambda g: (g - probs * g.sum(axis=axis, keepdims=True),), "log_softmax")

    def softmax(self, axis=-1):
        shifted = np.exp(self.data - self.data.max(axis=axis, keepdims=True))
        out = shifted / shifted.sum(axis=axis, keepdims=True)
        return self._result(out, (self,), lambda g: (out * (g - (g * out).sum(axis=axis, keepdims=True)),), "softmax")

    # reverse-mode differentiation
    def backward(self, grad=None):
        order, seen = [], set()

        def visit(node):
            if id(node) not in seen:
                seen.add(id(node))
                for parent in node._parents:
                    visit(parent)
                order.append(node)

        visit(self)
        self.grad = np.ones_like(self.data) if grad is None else np.asarray(grad, dtype=np.float64)
        for node in reversed(order):
            if node._backward is None or node.grad is None:
                continue
            for parent, parent_grad in zip(node._parents, node._backward(node.grad)):
                if not parent.requires_grad or parent_grad is None:
                    continue
                parent_grad = _unbroadcast(np.asarray(parent_grad, dtype=np.float64), parent.shape)
                parent.grad = parent_grad if parent.grad is None else parent.grad + parent_grad


def parameter(*shape, rng, scale=None):
    """A trainable weight, initialised with N(0, scale^2); default scale is He's sqrt(2 / fan_in)."""
    scale = np.sqrt(2.0 / shape[0]) if scale is None else scale
    return Tensor(rng.normal(0.0, scale, shape), requires_grad=True)


def zeros_parameter(*shape):
    return Tensor(np.zeros(shape), requires_grad=True)


def zero_grad(params):
    for p in params:
        p.grad = None


def cross_entropy(logits, targets, mask=None):
    """Mean next-token/class cross-entropy. logits: (..., classes); targets: int (...); mask: 1.0 keeps a position."""
    flat = logits.reshape(-1, logits.shape[-1]).log_softmax(axis=-1)
    targets = np.asarray(targets).reshape(-1)
    picked = flat[np.arange(targets.size), targets]
    if mask is None:
        return -picked.mean()
    weights = np.asarray(mask, dtype=np.float64).reshape(-1)
    return -(picked * weights).sum() / max(weights.sum(), 1.0)


def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    centred = x - mean
    var = (centred ** 2).mean(axis=-1, keepdims=True)
    return centred / (var + eps) ** 0.5 * gamma + beta


def numerical_grad(f, tensor, eps=1e-6):
    """Central-difference gradient of scalar f() with respect to tensor.data."""
    grad = np.zeros_like(tensor.data)
    it = np.nditer(tensor.data, flags=["multi_index"])
    for _ in it:
        index = it.multi_index
        original = tensor.data[index]
        tensor.data[index] = original + eps
        plus = float(f().data)
        tensor.data[index] = original - eps
        minus = float(f().data)
        tensor.data[index] = original
        grad[index] = (plus - minus) / (2 * eps)
    return grad


def relative_error(a, b):
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

## 📦 Recap: what you built earlier

The second hidden cell re-defines, under their usual names, the pieces from Base Model Training (`fit_standardiser`, `Linear`, `dropout`, `MLP`, `Adam`) and from notebook 1 of this course: `make_rare_class_data(n, seed, cut=2.8)`, `weighted_cross_entropy`, `positive_probability`, `precision_recall_curve`, `average_precision`, `threshold_for_recall(scores, y, min_recall)` (the minimum-recall rule from Exercise 1) and `confusion_counts`.

In [ ]:
# 📦 Recap: the pieces from Base Model Training and notebook 1, under their usual names.

def fit_standardiser(X):
    """Column means and scales of the TRAINING rows; apply them to every split as (X - mean) / scale."""
    return X.mean(axis=0), np.maximum(X.std(axis=0), 1e-6)


class Linear:
    """A dense layer x @ W + b, with He-initialised W of shape (n_in, n_out) and a zero bias."""

    def __init__(self, n_in, n_out, rng):
        self.W = parameter(n_in, n_out, rng=rng)
        self.b = zeros_parameter(n_out)

    def __call__(self, x):
        return x @ self.W + self.b

    def parameters(self):
        return [self.W, self.b]


def dropout(x, p, training, rng):
    """Inverted dropout: while training, zero each unit with probability p and scale the rest by 1 / (1 - p)."""
    if not training or p == 0.0:
        return x
    keep = (rng.random(x.shape) >= p) / (1.0 - p)
    return x * keep


class MLP:
    """ReLU hidden layers and a linear output layer, e.g. MLP([6, 32, 2], rng). Call it on a Tensor (batch, n_in)."""

    def __init__(self, sizes, rng, dropout=0.0):
        self.layers = [Linear(n_in, n_out, rng) for n_in, n_out in zip(sizes[:-1], sizes[1:])]
        self.p = dropout
        self.training = True

    def __call__(self, x, rng=None):
        for index, layer in enumerate(self.layers):
            x = layer(x)
            if index < len(self.layers) - 1:
                x = dropout(x.relu(), self.p, self.training, rng)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

    def train(self):
        self.training = True

    def eval(self):
        self.training = False


class Adam:
    """Adam over a list of engine parameters: opt.zero_grad(), loss.backward(), opt.step()."""

    def __init__(self, params, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params, self.lr, self.beta1, self.beta2, self.eps = list(params), lr, beta1, beta2, eps
        self.m = [np.zeros_like(p.data) for p in self.params]
        self.v = [np.zeros_like(p.data) for p in self.params]
        self.t = 0

    def zero_grad(self):
        zero_grad(self.params)

    def step(self):
        self.t += 1
        for p, m, v in zip(self.params, self.m, self.v):
            if p.grad is None:
                continue
            m[...] = self.beta1 * m + (1 - self.beta1) * p.grad
            v[...] = self.beta2 * v + (1 - self.beta2) * p.grad ** 2
            m_hat = m / (1 - self.beta1 ** self.t)
            v_hat = v / (1 - self.beta2 ** self.t)
            p.data -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


# From notebook 1 of this course.

def make_rare_class_data(n, seed, cut=2.8):
    """n rows of six standard-normal features; label 1 when a noisy score exceeds `cut` (about 11% of rows at 2.8)."""
    g = np.random.default_rng(seed)
    X = g.normal(size=(n, 6))
    score = 1.5 * X[:, 0] - X[:, 1] + X[:, 2] * X[:, 3] + 0.8 * g.normal(size=n)
    return X, (score > cut).astype(int)


def weighted_cross_entropy(logits, targets, class_weights):
    """Class-weighted mean cross-entropy as a scalar Tensor."""
    log_probs = logits.log_softmax(axis=-1)
    picked = log_probs[np.arange(len(targets)), targets]
    weights = class_weights[targets]
    return -(picked * weights).sum() / weights.sum()


def positive_probability(model, X):
    """P(positive) for every row of X: a NumPy array of shape (n,)."""
    model.eval()
    return model(Tensor(X)).softmax(axis=-1).data[:, 1]


def precision_recall_curve(scores, y):
    """Precision and recall for 'positive when score >= t', for every distinct score t (largest first)."""
    order = np.argsort(-scores, kind="stable")
    sorted_scores, sorted_y = scores[order], y[order]
    true_pos, false_pos = np.cumsum(sorted_y), np.cumsum(1 - sorted_y)
    last = np.r_[np.flatnonzero(np.diff(sorted_scores)), len(scores) - 1]
    tp, fp = true_pos[last], false_pos[last]
    return sorted_scores[last], tp / (tp + fp), tp / max(int(y.sum()), 1)


def average_precision(precision, recall):
    """Area under the step-shaped precision-recall curve."""
    return float(np.sum(np.diff(np.r_[0.0, recall]) * precision))


def threshold_for_recall(scores, y, min_recall):
    """The largest threshold with recall >= min_recall. Returns (threshold, precision, recall)."""
    thresholds, precision, recall = precision_recall_curve(scores, y)
    first = int(np.flatnonzero(recall >= min_recall)[0])
    return float(thresholds[first]), float(precision[first]), float(recall[first])


def confusion_counts(scores, y, threshold):
    """TP, FP, FN and TN of the rule 'positive when score >= threshold'."""
    predicted = scores >= threshold
    return {"tp": int(np.sum(predicted & (y == 1))), "fp": int(np.sum(predicted & (y == 0))),
            "fn": int(np.sum(~predicted & (y == 1))), "tn": int(np.sum(~predicted & (y == 0)))}

## 1 · Data parallelism in one paragraph

In **data-parallel** training, each of $k$ workers holds a complete **replica** of the model. Every global batch is split into $k$ shards, one per worker. Each worker runs forward and backward on its own shard and obtains the mean gradient over its rows. An **all-reduce** then combines the $k$ gradients into the gradient of the whole batch and hands the same result back to every worker, which applies the same update to its replica. The replicas start identical and receive identical updates, so they stay identical.

Two consequences matter. First, every worker still stores the whole model: data parallelism divides the *examples*, not the weights, so a model too big for one device needs model or state sharding instead. Second, the effective global batch is

$$\text{global batch} = \text{micro-batch} \times \text{accumulation steps} \times \text{workers},$$

assuming every worker sees the same number of examples per micro-batch.

The data are the rare-class splits from notebook 1.

In [ ]:
X_train_raw, y_train = make_rare_class_data(1000, seed=11)
X_val_raw, y_val = make_rare_class_data(500, seed=22)
X_test_raw, y_test = make_rare_class_data(500, seed=33)
mean, scale = fit_standardiser(X_train_raw)
X_train, X_val, X_test = [(X - mean) / scale for X in (X_train_raw, X_val_raw, X_test_raw)]

for workers, micro, accumulation in [(8, 4, 8), (4, 32, 1), (1, 16, 4)]:
    print(f"{workers} workers x micro-batch {micro} x {accumulation} accumulation steps = global batch {workers * micro * accumulation}")

## 2 · The all-reduce

Worker $i$ sees $n_i$ rows and computes the **mean** gradient $g_i$ over them. The gradient of the mean loss over all $N = \sum_i n_i$ rows is the example-weighted average

$$g = \frac{1}{N} \sum_{i} n_i \, g_i .$$

When every shard has the same size this is the plain average of the $g_i$, which is what frameworks such as PyTorch's `DistributedDataParallel` compute. When shards differ in size, a plain average gives each row of a small shard more say than each row of a large one. The cell shows the difference on a batch of 10 rows split over three workers as 4, 3 and 3 rows, with one number per row standing in for its gradient.

In [ ]:
per_row = np.arange(1.0, 11.0)  # stand-ins for ten per-row gradients
shards = np.array_split(per_row, 3)
print("shard sizes:", [len(s) for s in shards])
print("mean over all rows:            ", per_row.mean())
print("plain mean of the worker means:", np.mean([s.mean() for s in shards]))
print("weighted by rows per worker:   ", sum(len(s) * s.mean() for s in shards) / len(per_row))

### ✍️ Your turn: `all_reduce_mean`

Write `all_reduce_mean(worker_grads, counts)`:

- **inputs:** `worker_grads`, a list with one entry per worker, each a list of gradient arrays (one per parameter, in the same order for every worker); `counts`, the number of rows each worker saw;
- **output:** a list with one array per parameter: $\sum_i n_i\, g_i / \sum_i n_i$.

Hints: loop over the parameter index `j` and add up `count * grads[j]` over the workers; `zip(worker_grads, counts)` pairs each worker's gradients with its count.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def all_reduce_mean_reference(worker_grads, counts):
    total = sum(counts)
    return [sum(count * grads[j] for grads, count in zip(worker_grads, counts)) / total
            for j in range(len(worker_grads[0]))]

In [ ]:
def all_reduce_mean(worker_grads, counts):
    """Example-weighted average of the workers' gradients.
    worker_grads: one list of arrays per worker; counts: rows per worker. Returns one array per parameter."""
    # ✍️ Your code here (about three lines)
    raise NotImplementedError

In [ ]:
w1 = [np.array([[1.0, 2.0]]), np.array([0.5])]
w2 = [np.array([[3.0, -2.0]]), np.array([1.5])]
all_reduce_mean = practice("all_reduce_mean", all_reduce_mean, all_reduce_mean_reference, cases=[
    ([w1, w2], [2, 2]),
    ([w1, w2], [3, 1]),  # unequal shards: not the plain average
    ([[np.arange(4.0)] for _ in range(5)], [20, 19, 19, 19, 19]),  # 96 rows over 5 workers
    ([w1], [7]),  # edge case: a single worker
])

**Expected behaviour.** With equal counts the result is the plain average, `[[2, 0]]` and `[1]`. With counts 3 and 1 the first worker counts three times as much: `[[1.5, 1]]` and `[0.75]`. Identical gradients come back unchanged whatever the counts, and a single worker returns its own gradients.

## 3 · One data-parallel step, and a whole run

### ✍️ Your turn: `data_parallel_gradients`

Write `data_parallel_gradients(replicas, loss_fn, X, y)`, the simulated version of one step on $k$ workers:

- **inputs:** `replicas`, a list of $k$ models with identical weights; `loss_fn(model, X_part, y_part)`, which returns the mean loss of `model` on the rows it is given; the global batch `X`, `y`;
- **output:** the all-reduced gradient, one array per parameter.

Steps: split the row indices with `np.array_split(np.arange(len(y)), len(replicas))`; for each replica and its rows, call `zero_grad(model.parameters())`, run `loss_fn(...).backward()`, and keep `[p.grad.copy() for p in model.parameters()]` and the number of rows; finally return `all_reduce_mean(worker_grads, counts)`. Do not update any weights here.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference is in the collapsed cell below.

In [ ]:
def data_parallel_gradients_reference(replicas, loss_fn, X, y):
    worker_grads, counts = [], []
    for model, rows in zip(replicas, np.array_split(np.arange(len(y)), len(replicas))):
        zero_grad(model.parameters())
        loss_fn(model, X[rows], y[rows]).backward()
        worker_grads.append([p.grad.copy() for p in model.parameters()])
        counts.append(len(rows))
    return all_reduce_mean(worker_grads, counts)

In [ ]:
def data_parallel_gradients(replicas, loss_fn, X, y):
    """Shard the batch over the replicas, compute each worker's mean gradient, and all-reduce them."""
    worker_grads, counts = [], []
    # ✍️ Your code here (about six lines)
    raise NotImplementedError

In [ ]:
def mean_cross_entropy(model, X_part, y_part):
    return cross_entropy(model(Tensor(X_part)), y_part)


def replicas_of(sizes, seed, workers):
    """`workers` models built from the same seed, so their weights start identical."""
    return [MLP(sizes, np.random.default_rng(seed)) for _ in range(workers)]


X_ten, y_ten = X_train[:10], y_train[:10].copy()
y_ten[:2] = 1  # make sure both classes appear
data_parallel_gradients = practice("data_parallel_gradients", data_parallel_gradients, data_parallel_gradients_reference, cases=[
    (replicas_of([6, 4, 2], 0, 2), mean_cross_entropy, X_ten, y_ten),
    (replicas_of([6, 4, 2], 0, 3), mean_cross_entropy, X_ten, y_ten),  # shards of 4, 3 and 3 rows
    (replicas_of([6, 4, 2], 0, 1), mean_cross_entropy, X_ten, y_ten),  # edge case: one worker
])

Now whole runs. `train_data_parallel` keeps one replica and one Adam optimiser per worker. On every update it draws a global batch of 96 rows, computes the all-reduced gradient with your function, gives every replica that same gradient, and lets each optimiser step. The cell compares runs on 1, 4 and 5 workers: 96 rows split into shards of 24, or of 20 and 19.

In [ ]:
def train_data_parallel(workers, seed=0, updates=150, batch_size=96, sizes=(6, 32, 2)):
    """Simulated data-parallel training. Returns the list of replicas after training."""
    replicas = replicas_of(list(sizes), seed, workers)
    optimisers = [Adam(model.parameters(), lr=0.01) for model in replicas]
    g = np.random.default_rng(seed + 1)
    for _ in range(updates):
        rows = g.integers(0, len(y_train), size=batch_size)
        grads = data_parallel_gradients(replicas, mean_cross_entropy, X_train[rows], y_train[rows])
        for model, optimiser in zip(replicas, optimisers):
            for p, grad in zip(model.parameters(), grads):
                p.grad = grad.copy()
            optimiser.step()
    return replicas


def weights_of(model):
    return [p.data for p in model.parameters()]


def max_difference(model_a, model_b):
    return max(float(np.max(np.abs(a - b))) for a, b in zip(weights_of(model_a), weights_of(model_b)))


start = time.perf_counter()
single = train_data_parallel(1)
runs = {workers: train_data_parallel(workers) for workers in (4, 5)}
print(f"three runs in {time.perf_counter() - start:.1f}s")
for workers, replicas in runs.items():
    identical = all(max_difference(replicas[0], other) == 0.0 for other in replicas[1:])
    print(f"{workers} workers: replicas identical to each other: {identical}; "
          f"largest difference from the single-worker weights: {max_difference(replicas[0], single[0]):.1e}")

**Expected behaviour.** Within each run the replicas are identical to the last bit: they received the same gradient at every update. Compared with the single worker, the weights after 150 updates differ by about $10^{-16}$ to $10^{-15}$: the same arithmetic, added up in a different order. That is the whole correctness argument for data parallelism, and it is worth testing exactly like this, on one machine, before debugging a real multi-device run.

**Metrics need the same care.** Averaging per-worker metrics has the same trap as averaging gradients: workers with fewer rows get too much weight. Distributed samplers also often **pad** the data so that every worker gets the same number of rows, by repeating a few, and those duplicates must be removed before a validation metric is reported. The cell shows both on the 500 validation rows.

In [ ]:
model = single[0]
correct = (model(Tensor(X_val)).data.argmax(axis=1) == y_val).astype(float)
print(f"accuracy over all 500 validation rows: {correct.mean():.4f}")
uneven = [np.arange(0, 400), np.arange(400, 500)]  # say one worker received the 100 leftover rows
print(f"shards of 400 and 100 rows: plain mean of the two worker accuracies {np.mean([correct[r].mean() for r in uneven]):.4f}")
padded = np.r_[np.arange(500), np.arange(4)]  # 8 workers x 63 rows = 504, so four rows are repeated
print(f"8 workers with padding: {len(padded)} rows counted, accuracy {correct[padded].mean():.4f}; "
      f"after removing the duplicates {correct[np.unique(padded)].mean():.4f}")

The plain mean of two uneven shards drifts away from the accuracy over all rows, and the padded rows change the count from 500 to 504 while nudging the metric. Both effects grow with small validation sets and many workers. The safe habit is to remove duplicates, add up sums and counts across workers, and divide once.

## 4 · Reproducibility

A run is **reproducible** when repeating it gives the same result. "The same" can mean two different things:

- **bitwise**: identical bytes, which needs the same code, data order, seeds, library versions, hardware and order of floating-point operations;
- **within tolerance**: equal up to rounding, which is what you can expect after changing the number of workers, the hardware or the library version.

A **fingerprint**, a short hash of a model's exact bytes, makes the first kind cheap to check and to write down beside a result.

### ✍️ Your turn: `fingerprint`

Write `fingerprint(arrays)`:

- **input:** a list of NumPy arrays (for example a model's weights);
- **output:** the first 16 hexadecimal characters of a SHA-256 hash over every array, in order. For each array, feed the hash its shape as text, `str(a.shape).encode()`, and then its bytes as float64, `np.ascontiguousarray(a, dtype=np.float64).tobytes()`.

Including the shape means that `[1, 2]` and `[3]` do not hash like `[1]` and `[2, 3]`. Hints: `h = hashlib.sha256()`, `h.update(...)` for each piece, `h.hexdigest()[:16]`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference is in the collapsed cell below.

In [ ]:
def fingerprint_reference(arrays):
    h = hashlib.sha256()
    for a in arrays:
        h.update(str(np.shape(a)).encode())
        h.update(np.ascontiguousarray(a, dtype=np.float64).tobytes())
    return h.hexdigest()[:16]

In [ ]:
def fingerprint(arrays):
    """First 16 hex characters of SHA-256 over each array's shape and float64 bytes, in order."""
    # ✍️ Your code here (about five lines)
    raise NotImplementedError

In [ ]:
fingerprint = practice("fingerprint", fingerprint, fingerprint_reference, cases=[
    ([np.array([1.0, 2.0]), np.array([3.0])],),
    ([np.array([1.0]), np.array([2.0, 3.0])],),  # same numbers, different shapes: a different fingerprint
    ([np.arange(6).reshape(2, 3)],),             # integers are hashed as float64
    ([],),                                       # edge case: no arrays at all
])

Now the checks: the same seed twice, a different worker count, and a different seed. A run here has three sources of randomness, all seeded: the initial weights, the order of the batches and, for networks with dropout, the dropout masks.

In [ ]:
repeat = train_data_parallel(1)
other_seed = train_data_parallel(1, seed=7)
reference_print = fingerprint(weights_of(single[0]))
print(f"single worker, seed 0:        {reference_print}")
print(f"same run repeated:            {fingerprint(weights_of(repeat[0]))}  largest difference {max_difference(repeat[0], single[0]):.1e}")
print(f"4 workers, seed 0:            {fingerprint(weights_of(runs[4][0]))}  largest difference {max_difference(runs[4][0], single[0]):.1e}")
print(f"single worker, seed 7:        {fingerprint(weights_of(other_seed[0]))}  largest difference {max_difference(other_seed[0], single[0]):.1e}")

**Expected behaviour.** Repeating the run reproduces the fingerprint exactly. Four workers give weights that agree to about $10^{-15}$, so a tolerance check passes, while the fingerprint usually differs because a few bits did. A different seed gives a genuinely different model. So record the seed, the world size and the library versions next to a fingerprint, compare like with like bitwise, and compare across set-ups with a tolerance. The exact fingerprint strings depend on your computer and library versions, so do not expect to see the same ones as someone else.

## 5 · Capstone: a rare-class decision system

The capstone puts the course together on the rare-class task from notebook 1. The scaffold below runs end to end; your job is to make the choices marked ✏️, run it, and write the report at the end. Work in this order and do not skip ahead to the test set:

1. **Declare the objective** before looking at any candidate.
2. **Train a small, planned set of candidates**, choosing each one's checkpoint on validation data, and record every run in a ledger.
3. **Choose the winner and its threshold** on validation data with the declared objective.
4. **Save the complete decision rule**, reload it, and check that it reproduces the predictions.
5. **Evaluate once** on the test set, and report the counts with their uncertainty, the cost, and the variation across seeds.

### Step 1 · Declare the objective (✏️ capstone choices)

Suppose a missed positive, say a late delivery that nobody investigates, costs much more than a false alarm. A reasonable objective is then *the highest precision subject to finding at least 80% of the positives*. The rule `threshold_for_recall` from notebook 1 picks the largest threshold that meets the recall target on validation data, which raises the fewest alarms.

In [ ]:
REQUIRED_RECALL = 0.8  # ✏️ the recall the decision rule must reach on validation data
CANDIDATES = [         # ✏️ the planned candidates: change, add or remove, then rerun from here
    {"name": "plain, width 16", "width": 16, "weighting": "none"},
    {"name": "plain, width 32", "width": 32, "weighting": "none"},
    {"name": "weighted, width 16", "width": 16, "weighting": "balanced"},
    {"name": "weighted, width 32", "width": 32, "weighting": "balanced"},
]

### Step 2 · Train the candidates and keep a ledger

Every candidate gets the same protocol: 400 Adam updates of 128 rows, an evaluation every 25 updates, and the weights from the evaluation with the best validation **average precision** kept as its checkpoint. Average precision measures the ranking without committing to a threshold, so it suits checkpoint selection here. The threshold is then chosen on validation data with the declared objective, and the ledger records everything, including the runtime and the number of parameters.

In [ ]:
def train_candidate(candidate, seed=0, updates=400, every=25):
    """Train one candidate; keep the checkpoint with the best validation average precision."""
    start = time.perf_counter()
    model = MLP([6, candidate["width"], 2], np.random.default_rng(seed))
    weights = np.ones(2) if candidate["weighting"] == "none" else len(y_train) / (2 * np.bincount(y_train))
    optimiser, g = Adam(model.parameters(), lr=0.01), np.random.default_rng(seed + 1)
    best = (-1.0, None, 0)
    for update in range(1, updates + 1):
        rows = g.integers(0, len(y_train), size=128)
        model.train()
        optimiser.zero_grad()
        weighted_cross_entropy(model(Tensor(X_train[rows])), y_train[rows], weights).backward()
        optimiser.step()
        if update % every == 0:
            ap = average_precision(*precision_recall_curve(positive_probability(model, X_val), y_val)[1:])
            if ap > best[0]:
                best = (ap, [p.data.copy() for p in model.parameters()], update)
    for p, saved in zip(model.parameters(), best[1]):
        p.data[...] = saved
    threshold, precision, recall = threshold_for_recall(positive_probability(model, X_val), y_val, REQUIRED_RECALL)
    row = {"name": candidate["name"], "seed": seed, "best_update": best[2], "val_ap": best[0], "threshold": threshold,
           "val_precision": precision, "val_recall": recall, "parameters": sum(p.data.size for p in model.parameters()),
           "seconds": time.perf_counter() - start}
    return model, row


assert CANDIDATES, "add at least one candidate to CANDIDATES"
ledger, models = [], {}
for candidate in CANDIDATES:
    model, row = train_candidate(candidate)
    ledger.append(row)
    models[candidate["name"]] = model
print(f"{'candidate':>20} {'update':>6} {'val AP':>6} {'threshold':>9} {'precision':>9} {'recall':>6} {'params':>6} {'seconds':>7}")
for row in ledger:
    print(f"{row['name']:>20} {row['best_update']:6d} {row['val_ap']:6.3f} {row['threshold']:9.3f} "
          f"{row['val_precision']:9.3f} {row['val_recall']:6.3f} {row['parameters']:6d} {row['seconds']:7.2f}")

### Step 3 · Choose the winner on validation data

The declared objective decides: the highest validation precision at the required recall, with fewer parameters breaking a tie. Notice how close the candidates usually are. With about 50 validation positives, a difference of a few hundredths in precision is a handful of rows, so a simpler model that ties is a perfectly good choice.

In [ ]:
winner = max(ledger, key=lambda row: (row["val_precision"], -row["parameters"]))
winner_model = models[winner["name"]]
print(f"winner: {winner['name']} (validation precision {winner['val_precision']:.3f} at recall {winner['val_recall']:.3f}, "
      f"threshold {winner['threshold']:.3f})")

### Step 4 · Save the complete decision rule, and reload it

The evaluated system is not just the weights. It is the preprocessing, the weights and the threshold together: the same weights with a different threshold make different decisions, and the same weights without the standardiser see different inputs. The rule is saved with `np.savez` into an in-memory file, `io.BytesIO`, standing in for a file on disk, and loaded back.

In [ ]:
rule = {"mean": mean, "scale": scale, "threshold": winner["threshold"], "weights": [p.data.copy() for p in winner_model.parameters()]}


def save_rule(rule):
    buffer = io.BytesIO()
    arrays = {f"weight_{i}": w for i, w in enumerate(rule["weights"])}
    np.savez(buffer, mean=rule["mean"], scale=rule["scale"], threshold=np.array(rule["threshold"]), **arrays)
    return buffer.getvalue()


def load_rule(saved_bytes):
    with np.load(io.BytesIO(saved_bytes)) as data:
        count = sum(1 for key in data.files if key.startswith("weight_"))
        return {"mean": data["mean"], "scale": data["scale"], "threshold": float(data["threshold"]),
                "weights": [data[f"weight_{i}"] for i in range(count)]}


saved = save_rule(rule)
reloaded = load_rule(saved)
print(f"saved decision rule: {len(saved):,} bytes; keys: mean, scale, threshold and {len(reloaded['weights'])} weight arrays")

### ✍️ Your turn: `predict_with_rule`

Write `predict_with_rule(rule, X_raw)` in plain NumPy, the way a separate prediction service would use the saved file:

- **inputs:** a `rule` dict as above and **raw**, unstandardised inputs `X_raw` `(n, 6)`;
- **output:** an integer array `(n,)` of decisions, 1 where the positive-class probability is at least `rule["threshold"]`.

Steps: standardise with the rule's `mean` and `scale`; for each layer `i`, compute `h @ weights[2 * i] + weights[2 * i + 1]`, followed by `np.maximum(h, 0.0)` on every layer except the last; turn the two output columns into the probability of class 1 with a softmax (subtract each row's maximum first); compare with the threshold.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference is in the collapsed cell below.

In [ ]:
def predict_with_rule_reference(rule, X_raw):
    h = (X_raw - rule["mean"]) / rule["scale"]
    n_layers = len(rule["weights"]) // 2
    for i in range(n_layers):
        h = h @ rule["weights"][2 * i] + rule["weights"][2 * i + 1]
        if i < n_layers - 1:
            h = np.maximum(h, 0.0)
    exps = np.exp(h - h.max(axis=1, keepdims=True))
    return (exps[:, 1] / exps.sum(axis=1) >= rule["threshold"]).astype(int)

In [ ]:
def predict_with_rule(rule, X_raw):
    """Decisions (0 or 1) of a saved rule on raw inputs: standardise, run the layers, softmax, threshold."""
    # ✍️ Your code here (about eight lines)
    raise NotImplementedError

In [ ]:
toy_rule = {"mean": np.zeros(2), "scale": np.ones(2), "threshold": 0.5,
            "weights": [np.array([[1.0, -1.0], [0.5, 2.0]]), np.zeros(2), np.array([[1.0, 0.0], [0.0, 1.0]]), np.array([0.0, 0.1])]}
predict_with_rule = practice("predict_with_rule", predict_with_rule, predict_with_rule_reference, cases=[
    (toy_rule, np.array([[1.0, 0.0], [0.0, 1.0], [-1.0, -1.0]])),
    (reloaded, X_val_raw[:20]),
    (dict(toy_rule, threshold=1.01), np.array([[3.0, 3.0]])),  # edge case: a threshold above 1 never fires
])
in_memory = predict_with_rule(rule, X_val_raw)
after_reload = predict_with_rule(reloaded, X_val_raw)
engine_view = (positive_probability(winner_model, X_val) >= winner["threshold"]).astype(int)
print(f"validation decisions: in memory vs reloaded identical: {np.array_equal(in_memory, after_reload)}; "
      f"rows where the engine disagrees: {int(np.sum(engine_view != after_reload))}")

**Expected behaviour.** The reloaded rule makes exactly the same decisions as the rule in memory, and the plain-NumPy predictions agree with the engine's on every validation row (a row whose probability equals the threshold exactly could flip if your softmax rounds differently in the last bit). If a reload ever changes a decision, suspect a lost threshold, a different standardiser, or the weights of another checkpoint.

### Step 5 · Evaluate once, with uncertainty, cost and seeds

Everything is now fixed. The test set is used exactly once, through the reloaded rule. A recall measured on about 50 positives has a standard error of roughly $\sqrt{r(1-r)/P}$, which is why the counts and the number of positives belong in the report. Finally, the winning recipe is retrained with three more seeds, on training and validation data only, to see how much its validation result depends on luck.

In [ ]:
decisions = predict_with_rule(reloaded, X_test_raw)
tp = int(np.sum((decisions == 1) & (y_test == 1))); fp = int(np.sum((decisions == 1) & (y_test == 0)))
fn = int(np.sum((decisions == 0) & (y_test == 1))); tn = int(np.sum((decisions == 0) & (y_test == 0)))
test_recall, test_precision = tp / (tp + fn), tp / max(tp + fp, 1)
recall_se = np.sqrt(test_recall * (1 - test_recall) / (tp + fn))
print(f"test confusion counts: TP {tp}, FP {fp}, FN {fn}, TN {tn} ({tp + fn} positives among {len(y_test)} rows)")
print(f"test recall {test_recall:.3f} (standard error about {recall_se:.3f}), test precision {test_precision:.3f}")

winner_candidate = next(c for c in CANDIDATES if c["name"] == winner["name"])
seed_rows = [train_candidate(winner_candidate, seed=s)[1] for s in (1, 2, 3)]
precisions = [winner["val_precision"]] + [row["val_precision"] for row in seed_rows]
print(f"winner over 4 seeds: validation precision at recall {REQUIRED_RECALL} = {np.mean(precisions):.3f} "
      f"± {np.std(precisions, ddof=1):.3f} (range {min(precisions):.3f} to {max(precisions):.3f})")
print(f"cost: {winner['parameters']} parameters, {winner['seconds']:.2f}s to train on this computer; fingerprint {fingerprint(reloaded['weights'])}")

**What to notice.** The recall target was met on validation data, but test recall can land a few points below it: with about 50 positives, a few rows either way move it by several hundredths, and the threshold was chosen to *just* meet the target on validation data. If the requirement is strict, choose the threshold with a margin, or validate on more positives. The spread over seeds shows how much of the winner's lead over the other candidates is luck; compare it with the gaps in the ledger before claiming that one candidate is better.

### Your capstone report

Copy this outline into a new markdown cell and fill it in from your own run. Keep every number tied to the output above.

1. **Objective:** the recall target you declared and why; what a false positive and a false negative mean in the setting you imagine.
2. **Candidates and ledger:** every configuration you tried, including the ones that lost, with the checkpoint update, validation AP, threshold, precision, recall, parameters and seconds.
3. **Choice:** the winner and its threshold, chosen on validation data only, and the spread over seeds compared with the gaps between candidates.
4. **Artefact:** what the saved rule contains, and evidence that the reload reproduces the decisions.
5. **Test result:** the confusion counts, the number of positives, recall and precision with their uncertainty.
6. **Cost and failure:** parameters and training time; one hypothesis that did not pan out; where the rule still fails (the prevalence experiment is a good start).
7. **Limits:** the data are generated, and the threshold says nothing about any real population.

## 6 · 🧪 Experiments

### 🧪 Workers and shard sizes

Change the number of workers and the global batch, then run the cell again. The comparison is always against the single-worker run with the same batch.

In [ ]:
workers = 3      # 🧪 try 2, 7 and 16
batch_size = 96  # 🧪 try 100 (unequal shards for most worker counts)
baseline_run = train_data_parallel(1, batch_size=batch_size, updates=60)
parallel_run = train_data_parallel(workers, batch_size=batch_size, updates=60)
sizes = [len(r) for r in np.array_split(np.arange(batch_size), workers)]
print(f"{workers} workers, shards {sizes}: largest difference from one worker {max_difference(parallel_run[0], baseline_run[0]):.1e}")

Any number of workers reproduces the single-worker weights to rounding error, as long as the all-reduce weights each shard by its rows. With more workers than rows a shard would be empty and its mean gradient undefined, which real samplers avoid by padding or dropping rows.

### 🧪 A shift in prevalence

The threshold was chosen where about 10% of rows are positive. Generate test data with a different cut-off for the hidden score, which changes how common positives are (and, strictly, which rows count as positive), and apply the frozen rule.

In [ ]:
shifted_cut = 2.4  # 🧪 the original is 2.8; try 2.0 (more positives) and 3.4 (fewer)
X_shift, y_shift = make_rare_class_data(2000, seed=44, cut=shifted_cut)
d = predict_with_rule(reloaded, X_shift)
tp_s, fp_s, fn_s = int(np.sum((d == 1) & (y_shift == 1))), int(np.sum((d == 1) & (y_shift == 0))), int(np.sum((d == 0) & (y_shift == 1)))
print(f"cut {shifted_cut}: prevalence {y_shift.mean():.1%}, recall {tp_s / max(tp_s + fn_s, 1):.3f}, precision {tp_s / max(tp_s + fp_s, 1):.3f}")

When positives become more common, precision rises and recall falls; when they become rarer, precision falls. The model did not change, the world did: a threshold validated at one prevalence is a claim about that prevalence only. Monitoring the positive rate after deployment, and re-validating the threshold when it moves, is part of running such a system.

## 7 · Put it together

The cell checks the behaviour behind every claim in this notebook: the weighted all-reduce, equivalence of 1, 4 and 5 workers, bitwise reproducibility of a repeated run, and the capstone's protocol from ledger to reload to test. The protocol checks hold for any capstone choices; the quality floor on test is checked only for the default choices.

In [ ]:
batch_rows = np.arange(96)
full = replicas_of([6, 32, 2], 0, 1)[0]
zero_grad(full.parameters())
mean_cross_entropy(full, X_train[batch_rows], y_train[batch_rows]).backward()
full_batch = np.concatenate([p.grad.ravel() for p in full.parameters()])
five = data_parallel_gradients(replicas_of([6, 32, 2], 0, 5), mean_cross_entropy, X_train[batch_rows], y_train[batch_rows])
assert relative_error(np.concatenate([g.ravel() for g in five]), full_batch) < 1e-12, "5 workers must give the full-batch gradient"
for workers_run, replicas in runs.items():
    assert all(max_difference(replicas[0], other) == 0.0 for other in replicas[1:]), "replicas must stay identical"
    assert max_difference(replicas[0], single[0]) < 1e-10, f"{workers_run} workers must match one worker"
assert fingerprint(weights_of(repeat[0])) == reference_print, "a repeated run must reproduce the bytes"
assert max_difference(other_seed[0], single[0]) > 1e-3, "a different seed must give a different model"
assert len(ledger) == len(CANDIDATES), "the ledger must keep every candidate"
assert winner["val_recall"] >= REQUIRED_RECALL, "the chosen threshold must meet the recall target on validation data"
assert np.array_equal(in_memory, after_reload), "the reloaded rule must reproduce the decisions"
default_names = ["plain, width 16", "plain, width 32", "weighted, width 16", "weighted, width 32"]
if REQUIRED_RECALL == 0.8 and [c["name"] for c in CANDIDATES] == default_names:
    assert test_recall >= 0.55 and test_precision > 0.4, "with the default choices, the frozen rule should keep its quality on test"
else:
    print("Note: your capstone choices differ from the defaults, so the default quality floor on test is not checked.")
print("✅ Data parallelism, reproducibility and the capstone protocol all behave as expected.")
print(f"   capstone: {winner['name']}, threshold {winner['threshold']:.3f}; test recall {test_recall:.3f} ± {recall_se:.3f}, "
      f"precision {test_precision:.3f} ({tp + fn} test positives)")

## Exercises

### Exercise 1: count the updates

Write `updates_per_epoch(n_examples, micro_batch, accumulation_steps, workers, drop_last=True)` that returns how many optimiser updates one pass over `n_examples` makes, for the global batch $\text{micro} \times \text{accumulation} \times \text{workers}$. With `drop_last=True` an incomplete final global batch is dropped; otherwise it counts as one more update.

In [ ]:
def updates_per_epoch_reference(n_examples, micro_batch, accumulation_steps, workers, drop_last=True):
    global_batch = micro_batch * accumulation_steps * workers
    full, rest = divmod(n_examples, global_batch)
    return full + (0 if drop_last or rest == 0 else 1)

In [ ]:
def updates_per_epoch(n_examples, micro_batch, accumulation_steps, workers, drop_last=True):
    """Optimiser updates in one pass over the data for the global batch micro x accumulation x workers."""
    # ✍️ Your code here (about three lines)
    raise NotImplementedError

In [ ]:
updates_per_epoch = practice("updates_per_epoch", updates_per_epoch, updates_per_epoch_reference, cases=[
    (8192, 16, 4, 1), (8192, 4, 8, 8), (1000, 16, 4, 2, False), (1000, 16, 4, 2, True), (100, 64, 1, 1),
])

### Exercise 2: a padded validation metric

A distributed sampler hands each worker the row ids `ids` and the per-row results `values`, with some rows repeated to pad the shards. Write `deduplicated_mean(values, ids)` that averages each distinct row once (keep the first occurrence of each id).

In [ ]:
def deduplicated_mean_reference(values, ids):
    _, first = np.unique(ids, return_index=True)
    return float(np.mean(np.asarray(values)[first]))

In [ ]:
def deduplicated_mean(values, ids):
    """Mean of `values` counting each distinct id once (its first occurrence)."""
    # ✍️ Your code here (two lines)
    raise NotImplementedError

In [ ]:
deduplicated_mean = practice("deduplicated_mean", deduplicated_mean, deduplicated_mean_reference, cases=[
    (np.array([1.0, 0.0, 1.0, 1.0]), np.array([0, 1, 2, 0])),
    (np.array([0.5, 0.25, 0.75]), np.array([2, 1, 0])),
    (np.array([1.0, 1.0, 0.0]), np.array([4, 4, 4])),  # edge case: one row, padded twice
])

### Exercise 3 (question): resuming after a failure

A worker crashes after update 3,000 of a long data-parallel run. List everything the checkpoint must contain so that the resumed run continues exactly as if nothing had happened: think of the weights, the optimiser's moments and step count, the learning-rate schedule's position, the data order and every random generator. Which of these does `rule` in Step 4 deliberately leave out, and why is that right for a decision rule but wrong for resuming training?

### Exercise 4 (question): when data parallelism is not enough

A model's weights, gradients and Adam moments need 40 GB, and each device has 24 GB. Explain why adding data-parallel workers cannot make it fit, and what model or state sharding changes instead.

## Common errors and debugging

- **A plain average over unequal shards.** Weight each worker's mean gradient, or each worker's metric, by its rows; or add up sums and counts and divide once.
- **Replicas that drift apart.** Every replica must start from the same weights and apply the same all-reduced gradient. Different initial seeds on different workers are a classic bug.
- **Expecting bitwise equality across set-ups.** A different worker count, device or library changes the order of additions; compare with a tolerance and record the set-up.
- **Expecting data parallelism to fit a bigger model.** Each worker still holds every weight; it divides examples, not memory for the model.
- **Duplicated validation rows.** Padding samplers repeat rows; remove the duplicates before reporting a metric.
- **Saving only the weights.** The decision rule is preprocessing plus weights plus threshold; a reload without one of them is a different system.
- **Re-choosing the threshold after seeing the test results.** That turns the test set into a second validation set, and the reported numbers become optimistic.
- **Reporting the best seed.** Report the spread over seeds next to the differences you claim.

## Summary

Simulated workers split each batch, computed their own mean gradients and combined them with an example-weighted all-reduce; with that one rule, 1, 4 and 5 workers trained the same model up to rounding error, and the replicas never drifted apart. A fingerprint showed when two runs are the same to the bit, and a tolerance showed when they are the same up to the order of additions. The capstone then ran the whole course's protocol: a declared objective, a ledger of candidates, checkpoint and threshold chosen on validation data, a saved and reloaded decision rule, and a single test evaluation reported with its uncertainty, its cost and its variation over seeds. Everything here ran on generated data and in one process: real distributed training adds devices, networks, failures and far larger models, and real decision systems face shifting populations that this test set cannot represent.

## Next

**Transformers**: the next course builds attention and a small transformer with the same tiny tensor engine, and the training controls from this course, schedules, clipping, accumulation and careful selection, carry straight over.